# 🗺️ Oefening: Okada Fault Modeling using cutde (Bali / Flores Thrust)

Pas in de cel hieronder de variabelen aan in **Sectie 1** (zoals `dip` of `slip`) en druk op **Shift + Enter** om de deformatiekaart opnieuw te berekenen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cutde.fullspace as FS

print("🔄 Starting fault calculations using cutde...")

# ==============================================================================
# 1. FAULT PARAMETERS (Edit numbers here)
# ==============================================================================
strike = 90.0       # Fault direction (degrees clockwise from North)
dip = 30.0          # Fault dip angle (degrees)
rake = 90.0         # Motion direction: 90 = Pure Thrust/Reverse
slip = 5.0          # Total fault displacement during earthquake (meters)

fault_depth = 5.0   # Depth to the top edge of the fault plane (km)
fault_length = 30.0 # Length of the fault along its strike (km)
fault_width = 15.0  # Width of the fault down its dip plane (km)
poisson_ratio = 0.25

# ==============================================================================
# 2. GEOMETRY SETUP
# ==============================================================================
x_coords = np.linspace(-40, 40, 50)
y_coords = np.linspace(-40, 40, 50)
X, Y = np.meshgrid(x_coords, y_coords)

# Reshape observation points to a N x 3 array [x, y, z] for cutde
obs_pts = np.vstack([X.flatten(), Y.flatten(), np.zeros_like(X.flatten())]).T

# Convert geometric properties to radians
dip_rad = np.deg2rad(dip)
rake_rad = np.deg2rad(rake)
u_strike = slip * np.cos(rake_rad)
u_dip = slip * np.sin(rake_rad)

# Define the 4 corners of the rectangular fault plane in 3D space
d_top = fault_depth
d_bottom = fault_depth + fault_width * np.sin(dip_rad)
w_proj = fault_width * np.cos(dip_rad)

fault_pts = np.array([
    [-fault_length/2, 0.0, -d_top],
    [fault_length/2, 0.0, -d_top],
    [fault_length/2, w_proj, -d_bottom],
    [-fault_length/2, w_proj, -d_bottom]
])

# Define the two triangles that form the rectangular fault plane
tris = np.array([[0, 1, 2], [0, 2, 3]])
fault_disps = np.array([[u_strike, u_dip, 0.0], [u_strike, u_dip, 0.0]])

# ==============================================================================
# 3. RUN MATHEMATICAL DISLOCATION CALCULATIONS
# ==============================================================================
disp = FS.disp(obs_pts, fault_pts[tris], fault_disps, poisson_ratio)
Z_uplift = disp[:, 2].reshape(X.shape)  # Extract vertical displacement
print("✅ Calculation successfully completed!")

# ==============================================================================
# 4. PLOT GENERATION
# ==============================================================================
plt.figure(figsize=(9, 7))
contour = plt.contourf(X, Y, Z_uplift, levels=20, cmap='RdBu_r')
cbar = plt.colorbar(contour)
cbar.set_label('Vertical Surface Displacement / Uplift (meters)')
plt.axhline(0, color='black', linestyle='--', linewidth=2, label='Fault Axis Trace')
plt.title('Okada Surface Deformation Map (via cutde)')
plt.xlabel('Distance Perpendicular to Strike (km)')
plt.ylabel('Distance Parallel to Strike (km)')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()